# Statistical Verification of n Parameter vs Looping

This notebook verifies that OpenAI's `n` parameter produces the same statistical distribution as making separate API calls in a loop.

We'll test this by asking GPT-5-mini to generate random numbers between 1-100 and comparing the distributions.

In [ ]:
import openai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
import os
from tqdm import tqdm
import time

In [ ]:
# Set up OpenAI client
client = openai.OpenAI()

# Model to test
MODEL = "gpt-5-mini"  # Using the new GPT-5-mini model

## Test 1: Generate Random Numbers with n Parameter

In [ ]:
def get_random_number_n_parameter(n=100):
    """Get random numbers using the n parameter in a single API call."""
    
    prompt = "Generate a random integer between 1 and 100 (inclusive). Reply with just the number, nothing else."
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            n=n,
            temperature=1.0,
            max_tokens=10
        )
        
        numbers = []
        for choice in response.choices:
            try:
                num = int(choice.message.content.strip())
                if 1 <= num <= 100:
                    numbers.append(num)
            except:
                pass  # Skip invalid responses
        
        return numbers
    except Exception as e:
        print(f"Error with n={n}: {e}")
        return []

In [ ]:
# Test with n parameter
print("Testing with n parameter...")
start_time = time.time()

# Get 100 samples using n=100
numbers_n_param = get_random_number_n_parameter(n=100)

n_param_time = time.time() - start_time
print(f"Generated {len(numbers_n_param)} numbers in {n_param_time:.2f} seconds")
print(f"Sample: {numbers_n_param[:10]}")

## Test 2: Generate Random Numbers with Loop

In [ ]:
def get_random_number_single():
    """Get a single random number with a separate API call."""
    
    prompt = "Generate a random integer between 1 and 100 (inclusive). Reply with just the number, nothing else."
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=1.0,
            max_tokens=10
        )
        
        num = int(response.choices[0].message.content.strip())
        if 1 <= num <= 100:
            return num
    except:
        pass
    return None

In [ ]:
# Test with loop
print("Testing with loop (100 separate API calls)...")
start_time = time.time()

numbers_loop = []
for i in tqdm(range(100)):
    num = get_random_number_single()
    if num is not None:
        numbers_loop.append(num)
    time.sleep(0.1)  # Small delay to avoid rate limits

loop_time = time.time() - start_time
print(f"Generated {len(numbers_loop)} numbers in {loop_time:.2f} seconds")
print(f"Sample: {numbers_loop[:10]}")

## Statistical Analysis

In [ ]:
# Basic statistics
stats_df = pd.DataFrame({
    'Method': ['n Parameter', 'Loop'],
    'Count': [len(numbers_n_param), len(numbers_loop)],
    'Mean': [np.mean(numbers_n_param), np.mean(numbers_loop)],
    'Std': [np.std(numbers_n_param), np.std(numbers_loop)],
    'Min': [np.min(numbers_n_param), np.min(numbers_loop)],
    'Max': [np.max(numbers_n_param), np.max(numbers_loop)],
    'Time (s)': [n_param_time, loop_time]
})

print("\nBasic Statistics:")
print(stats_df.to_string(index=False))

# Speed improvement
speedup = loop_time / n_param_time
print(f"\nSpeed improvement with n parameter: {speedup:.1f}x faster")

In [ ]:
# Statistical tests
print("\n=== Statistical Tests ===")

# 1. Kolmogorov-Smirnov test (tests if two samples come from same distribution)
ks_stat, ks_pvalue = stats.ks_2samp(numbers_n_param, numbers_loop)
print(f"\nKolmogorov-Smirnov Test:")
print(f"  Statistic: {ks_stat:.4f}")
print(f"  P-value: {ks_pvalue:.4f}")
print(f"  Result: {'Same distribution' if ks_pvalue > 0.05 else 'Different distributions'} (α=0.05)")

# 2. Mann-Whitney U test (tests if distributions have same median)
mw_stat, mw_pvalue = stats.mannwhitneyu(numbers_n_param, numbers_loop, alternative='two-sided')
print(f"\nMann-Whitney U Test:")
print(f"  Statistic: {mw_stat:.4f}")
print(f"  P-value: {mw_pvalue:.4f}")
print(f"  Result: {'Same median' if mw_pvalue > 0.05 else 'Different medians'} (α=0.05)")

# 3. Chi-square test for uniformity
# Create frequency bins
bins = np.arange(1, 102, 10)  # 10 bins
hist_n, _ = np.histogram(numbers_n_param, bins=bins)
hist_loop, _ = np.histogram(numbers_loop, bins=bins)

# Chi-square test comparing the two distributions
chi2_stat, chi2_pvalue = stats.chisquare(hist_n + 1, hist_loop + 1)  # Add 1 to avoid zero counts
print(f"\nChi-Square Test (comparing distributions):")
print(f"  Statistic: {chi2_stat:.4f}")
print(f"  P-value: {chi2_pvalue:.4f}")
print(f"  Result: {'Same distribution' if chi2_pvalue > 0.05 else 'Different distributions'} (α=0.05)")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Histogram comparison
axes[0, 0].hist(numbers_n_param, bins=20, alpha=0.5, label='n Parameter', color='blue', edgecolor='black')
axes[0, 0].hist(numbers_loop, bins=20, alpha=0.5, label='Loop', color='red', edgecolor='black')
axes[0, 0].set_xlabel('Random Number')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution Comparison')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Q-Q Plot
stats.probplot(numbers_n_param, dist="uniform", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot: n Parameter vs Uniform')

# Box plot comparison
axes[1, 0].boxplot([numbers_n_param, numbers_loop], labels=['n Parameter', 'Loop'])
axes[1, 0].set_ylabel('Random Number')
axes[1, 0].set_title('Box Plot Comparison')
axes[1, 0].grid(True, alpha=0.3)

# Cumulative distribution
axes[1, 1].hist(numbers_n_param, bins=50, cumulative=True, alpha=0.5, label='n Parameter', density=True, color='blue')
axes[1, 1].hist(numbers_loop, bins=50, cumulative=True, alpha=0.5, label='Loop', density=True, color='red')
axes[1, 1].set_xlabel('Random Number')
axes[1, 1].set_ylabel('Cumulative Probability')
axes[1, 1].set_title('Cumulative Distribution')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Test for Independence within n Parameter Results

In [ ]:
# Test if samples within n parameter are independent
# We'll check for autocorrelation
from statsmodels.stats.diagnostic import acorr_ljungbox

# Ljung-Box test for autocorrelation
lb_result = acorr_ljungbox(numbers_n_param, lags=10, return_df=True)
print("\nLjung-Box Test for Independence (n parameter samples):")
print(lb_result[['lb_pvalue']].T)
print(f"\nAll p-values > 0.05: {all(lb_result['lb_pvalue'] > 0.05)}")
print("Result: Samples are", "independent" if all(lb_result['lb_pvalue'] > 0.05) else "not independent")

## Extended Test with Multiple Runs

In [ ]:
# Run multiple batches to get more robust statistics
print("Running extended test with multiple batches...\n")

n_batches = 5
batch_size = 50

all_n_param = []
all_loop = []

for batch in range(n_batches):
    print(f"Batch {batch + 1}/{n_batches}")
    
    # n parameter method
    batch_n = get_random_number_n_parameter(n=batch_size)
    all_n_param.extend(batch_n)
    
    # Loop method
    batch_loop = []
    for _ in range(batch_size):
        num = get_random_number_single()
        if num is not None:
            batch_loop.append(num)
        time.sleep(0.05)  # Small delay
    all_loop.extend(batch_loop)
    
    time.sleep(1)  # Pause between batches

print(f"\nTotal samples - n parameter: {len(all_n_param)}, Loop: {len(all_loop)}")

In [ ]:
# Final statistical comparison
print("\n=== Final Statistical Comparison ===")

# KS test on larger sample
ks_stat_final, ks_pvalue_final = stats.ks_2samp(all_n_param, all_loop)
print(f"\nKolmogorov-Smirnov Test (Extended):")
print(f"  Statistic: {ks_stat_final:.4f}")
print(f"  P-value: {ks_pvalue_final:.4f}")
print(f"  Result: {'✓ Same distribution' if ks_pvalue_final > 0.05 else '✗ Different distributions'} (α=0.05)")

# Summary statistics
print(f"\nSummary Statistics (Extended):")
print(f"  n Parameter - Mean: {np.mean(all_n_param):.2f}, Std: {np.std(all_n_param):.2f}")
print(f"  Loop       - Mean: {np.mean(all_loop):.2f}, Std: {np.std(all_loop):.2f}")
print(f"  Expected (uniform 1-100) - Mean: 50.5, Std: 28.87")

# Test against theoretical uniform distribution
print(f"\nTest Against Uniform Distribution:")
_, p_uniform_n = stats.kstest(all_n_param, lambda x: x/100)
_, p_uniform_loop = stats.kstest(all_loop, lambda x: x/100)
print(f"  n Parameter: p-value = {p_uniform_n:.4f}")
print(f"  Loop:        p-value = {p_uniform_loop:.4f}")

## Conclusion

Based on the statistical tests above, we can determine whether the `n` parameter produces the same distribution as looping through separate API calls. Key findings:

1. **Performance**: The n parameter is significantly faster (expected ~100x for n=100)
2. **Distribution**: Statistical tests show whether the distributions are equivalent
3. **Independence**: Tests verify if samples within n parameter calls are independent
4. **Uniformity**: Both methods should produce approximately uniform distributions for random numbers